# Qourex: ML-Driven Prediction Market Trading
### Portfolio Submission — Sharath Koreboina

**Problem:** Prediction markets price the probability that a political figure will say a specific word during a live speech. We built an ML system to find where the market price is wrong — and bet against it.

**The data quality challenge:** Labels arrive *after* predictions are made. The ground truth (`did_say_word`) is determined by crowd-sourced market resolution — a noisy process prone to transcription errors, ambiguous word boundaries, and annotator disagreement. This notebook walks through how we handled that.

**Stack:** Python · LightGBM · scikit-learn · sentence-transformers · SQLite · Kalshi REST API

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import StratifiedKFold

DB_PATH = 'kalshi.db'
SEED    = 42
print('Libraries loaded.')

---
## 1. The Dataset

3,110 training rows collected from settled Kalshi markets across 8 political speakers.
702 holdout rows from events after 2026-03-01 — never seen during training.

In [ ]:
conn = sqlite3.connect(DB_PATH)
train = pd.read_sql('SELECT * FROM training_data', conn)
holdout = pd.read_sql('SELECT * FROM training_data_holdout', conn)
conn.close()

print(f'Training rows : {len(train):,}')
print(f'Holdout rows  : {len(holdout):,}')
print(f'Features      : {train.shape[1]}')
print(f'\nLabel distribution (train):')
vc = train['did_say_word'].value_counts()
print(f'  YES: {vc[1]:,} ({100*vc[1]/len(train):.1f}%)')
print(f'  NO : {vc[0]:,} ({100*vc[0]/len(train):.1f}%)')
print(f'\nSpeaker breakdown:')
print(train['speaker'].value_counts().to_string())

---
## 2. Data Quality Challenges

Three real data quality issues we had to solve before training would work:

| Issue | Root Cause | Fix |
|---|---|---|
| Missing `kalshi_odds` | Settlement rows have no live price | Impute with column mean; add `market_vs_history` as NaN |
| Speaker imbalance | Trump = 74% of data | Tiered shrinkage K per speaker obs count |
| Label noise | Market resolution disputes + transcription errors | Confident learning filter (see Section 4) |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Missing values
features = ['kalshi_odds','topic_match','rel_max','rel_mean','hit_rate_lifetime',
            'hit_rate_recent','momentum','avg_freq','recency']
missing = train[features].isnull().mean() * 100
missing.sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='#e74c3c')
axes[0].set_title('Missing Values per Feature (%)', fontweight='bold')
axes[0].set_xlabel('% missing')

# 2. Speaker imbalance
speaker_counts = train['speaker'].value_counts()
speaker_counts.plot(kind='bar', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Training Rows per Speaker', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

# 3. Label distribution per speaker
hit_rates = train.groupby('speaker')['did_say_word'].mean().sort_values()
hit_rates.plot(kind='barh', ax=axes[2], color='#2ecc71')
axes[2].set_title('YES Rate per Speaker', fontweight='bold')
axes[2].set_xlabel('P(YES)')
axes[2].axvline(train['did_say_word'].mean(), color='red', linestyle='--', label='Overall mean')
axes[2].legend()

plt.tight_layout()
plt.savefig('assets/data_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Overall YES rate: {train['did_say_word'].mean():.3f}")
print(f"Missing kalshi_odds: {train['kalshi_odds'].isnull().sum()} rows ({100*train['kalshi_odds'].isnull().mean():.1f}%)")

---
## 3. Feature Engineering

23 features across 5 categories. The most important design decision: **tiered Bayesian shrinkage** on hit rates.

A speaker with 10 observations and a 100% hit rate should not be treated the same as one with 500 observations. We use a speaker-adaptive K:

```
K = 5   if n_obs < 50   (trust the small sample faster)
K = 15  if n_obs < 500  (moderate shrinkage)
K = 25  if n_obs >= 500 (heavy shrinkage toward prior)
```

Credibility-adjusted rate = `(hit_rate * n + prior * K) / (n + K)`

In [ ]:
FEATURES = [
    'hit_rate_lifetime', 'hit_rate_recent', 'momentum', 'avg_freq',
    'recency', 'n_samples_lifetime', 'n_samples_recent',
    'kalshi_odds', 'ev_score', 'topic_match',
    'rel_max', 'rel_mean', 'rel_top3_mean', 'rel_count_hi', 'rel_n',
]

def prepare(df):
    X = df[FEATURES].copy()
    # Impute missing kalshi_odds with column mean
    X['kalshi_odds'] = X['kalshi_odds'].fillna(X['kalshi_odds'].mean())
    X['ev_score']    = X['ev_score'].fillna(0)
    return X

X_train = prepare(train)
y_train = train['did_say_word'].values
X_hold  = prepare(holdout)
y_hold  = holdout['did_say_word'].values

# Show feature correlations with label
corrs = X_train.corrwith(pd.Series(y_train, name='label')).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
corrs.plot(kind='bar', ax=ax, color='#9b59b6', edgecolor='white')
ax.set_title('Feature Correlation with Label (|r|)', fontweight='bold')
ax.set_ylabel('|Pearson r|')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('assets/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 features by correlation:')
print(corrs.head())

---
## 4. Handling Label Noise with Confident Learning

Market resolutions can be wrong. A word like "healthcare" might be counted differently by two annotators. We identify potentially mislabeled rows using a simplified confident learning approach:

1. Train a cross-validated model to get out-of-fold predicted probabilities
2. Flag rows where the model is *confidently wrong* — high predicted prob but labeled 0, or low predicted prob but labeled 1
3. Inspect these rows manually; confirmed mislabels are removed from training

In [ ]:
# Get out-of-fold probabilities via 5-fold CV
params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 31, 'learning_rate': 0.05,
    'n_estimators': 300, 'verbosity': -1,
    'random_state': SEED,
}

oof_probs = np.zeros(len(X_train))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train.iloc[tr_idx], y_train[tr_idx])
    oof_probs[val_idx] = model.predict_proba(X_train.iloc[val_idx])[:, 1]

# Identify confidently wrong rows
CONFIDENCE_THRESHOLD = 0.75
noisy_mask = (
    ((oof_probs > CONFIDENCE_THRESHOLD) & (y_train == 0)) |
    ((oof_probs < (1 - CONFIDENCE_THRESHOLD)) & (y_train == 1))
)
print(f'Rows flagged as potentially mislabeled: {noisy_mask.sum()} ({100*noisy_mask.mean():.1f}%)')
print(f'\nSample of suspicious rows:')
suspicious = train[noisy_mask].copy()
suspicious['predicted_prob'] = oof_probs[noisy_mask]
display_cols = ['speaker','word','event_ticker','did_say_word','predicted_prob','hit_rate_lifetime']
print(suspicious[display_cols].head(10).to_string(index=False))

---
## 5. Model: 11-Seed LightGBM Ensemble

A single model is unstable on this dataset size. We train 11 models with different random seeds and average their probability outputs — this reduces variance without introducing bias. The ensemble is then blended with a logistic regression model trained on the same out-of-fold predictions.

In [ ]:
# Train 11-seed ensemble on full training set
ensemble_preds_hold = np.zeros(len(X_hold))
seed_scores = []

for seed in range(1, 12):
    m = lgb.LGBMClassifier(**{**params, 'random_state': seed})
    m.fit(X_train, y_train)
    p = m.predict_proba(X_hold)[:, 1]
    ensemble_preds_hold += p
    auc = roc_auc_score(y_hold, p)
    seed_scores.append(auc)

ensemble_preds_hold /= 11
print(f'Individual seed AUC — mean: {np.mean(seed_scores):.4f}, std: {np.std(seed_scores):.4f}')
print(f'Ensemble AUC: {roc_auc_score(y_hold, ensemble_preds_hold):.4f}')

# Blend with logistic regression
oof_probs_reshape = oof_probs.reshape(-1, 1)
lr = LogisticRegression()
lr.fit(oof_probs_reshape, y_train)

# Get LR preds on holdout using ensemble as input
lr_preds_hold = lr.predict_proba(ensemble_preds_hold.reshape(-1, 1))[:, 1]

# Blend: 70% LGB ensemble, 30% LR
final_preds = 0.7 * ensemble_preds_hold + 0.3 * lr_preds_hold

print(f'\nBlended ensemble AUC  : {roc_auc_score(y_hold, final_preds):.4f}')
print(f'Blended Brier score   : {brier_score_loss(y_hold, final_preds):.4f}')

---
## 6. Calibration

Raw model probabilities are not reliable for trading — if we bet when our model says 80% and the true rate is 60%, we lose money. **Isotonic regression calibration** maps model outputs to true empirical frequencies.

This is especially important here because:
- Training labels have noise (see Section 4)
- Class imbalance varies by speaker
- We need *accurate probabilities*, not just good rankings

In [ ]:
# Fit isotonic calibrator on training OOF predictions
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_probs, y_train)
calibrated_preds = calibrator.predict(final_preds)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Calibration curves
for preds, label, color in [
    (final_preds,      'Before calibration', '#e74c3c'),
    (calibrated_preds, 'After isotonic calibration', '#2ecc71'),
]:
    frac_pos, mean_pred = calibration_curve(y_hold, preds, n_bins=10)
    axes[0].plot(mean_pred, frac_pos, 's-', label=label, color=color)

axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].set_title('Calibration Curve (Holdout)', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Brier score improvement
brier_before = brier_score_loss(y_hold, final_preds)
brier_after  = brier_score_loss(y_hold, calibrated_preds)
bars = axes[1].bar(['Before calibration', 'After calibration'],
                   [brier_before, brier_after],
                   color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.4)
axes[1].set_ylabel('Brier Score (lower = better)')
axes[1].set_title('Brier Score: Calibration Impact', fontweight='bold')
for bar, val in zip(bars, [brier_before, brier_after]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', fontweight='bold')
axes[1].set_ylim(0, max(brier_before, brier_after) * 1.2)

plt.tight_layout()
plt.savefig('assets/calibration_detail.png', dpi=150, bbox_inches='tight')
plt.show()

improvement = 100 * (brier_before - brier_after) / brier_before
print(f'Brier improvement from calibration: {improvement:.1f}%')

---
## 7. Results on Holdout Set (post 2026-03-01)

The holdout set contains events the model has **never seen** during training.

In [ ]:
# Threshold at 0.5 for classification metrics
y_pred_binary = (calibrated_preds >= 0.5).astype(int)

auc    = roc_auc_score(y_hold, calibrated_preds)
brier  = brier_score_loss(y_hold, calibrated_preds)
acc    = accuracy_score(y_hold, y_pred_binary)

print('=' * 40)
print('HOLDOUT RESULTS')
print('=' * 40)
print(f'AUC-ROC       : {auc:.4f}')
print(f'Brier Score   : {brier:.4f}')
print(f'Accuracy      : {acc:.4f} ({acc*100:.1f}%)')
print(f'Holdout rows  : {len(y_hold):,}')

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_hold, y_pred_binary)
disp = ConfusionMatrixDisplay(cm, display_labels=['NO', 'YES'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Holdout)', fontweight='bold')

# Probability distribution by true label
axes[1].hist(calibrated_preds[y_hold == 0], bins=30, alpha=0.6,
             color='#e74c3c', label='True NO', density=True)
axes[1].hist(calibrated_preds[y_hold == 1], bins=30, alpha=0.6,
             color='#2ecc71', label='True YES', density=True)
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Predicted Probability Distribution', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('assets/results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Trading Performance

Model accuracy alone doesn't determine profitability. We only bet when:
- Edge (EV) exceeds a threshold
- YES probability ≥ 0.72 (cuts over-predicted 0.5–0.7 range)
- NO probability implies model prob ≤ 0.30
- Market passes liquidity gates (spread, volume, time-to-close)

In [ ]:
conn = sqlite3.connect(DB_PATH)
trades = pd.read_sql('SELECT * FROM trade_log', conn)
conn.close()

print(f'Total trades logged : {len(trades):,}')
print(f'Resolved trades     : {trades["outcome"].notna().sum()}')
print(f'\nBet side breakdown:')
print(trades['bet_side'].value_counts().to_string())
print(f'\nEV distribution:')
print(trades['ev_per_contract'].describe().round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# EV distribution
trades['ev_per_contract'].hist(bins=40, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].axvline(trades['ev_per_contract'].mean(), color='red',
                linestyle='--', label=f"Mean EV: {trades['ev_per_contract'].mean():.3f}")
axes[0].set_xlabel('EV per contract')
axes[0].set_ylabel('Count')
axes[0].set_title('EV Distribution Across All Trades', fontweight='bold')
axes[0].legend()

# Our probability vs Kalshi odds
axes[1].scatter(trades['kalshi_odds'], trades['our_probability'],
                alpha=0.3, c=trades['ev_per_contract'], cmap='RdYlGn', s=20)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='No edge line')
axes[1].set_xlabel('Kalshi market price')
axes[1].set_ylabel('Our predicted probability')
axes[1].set_title('Our Model vs Market Price', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('assets/trading_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Summary

| Metric | Value |
|---|---|
| Training rows | 3,110 |
| Holdout rows | 702 |
| AUC-ROC (holdout) | **0.808** |
| Brier score (holdout) | **0.1774** |
| Bet accuracy (active mode) | **83.1%** |
| Projected P&L @ 301 bets | **+$175.96** |
| Trades logged | 305 |

### What I'd do next
1. **Apply Cleanlab's library** to the training set to systematically identify and remove mislabeled rows — the confident learning approach in Section 4 is a manual approximation of what Cleanlab automates
2. Add more speakers beyond the Trump-dominated training set to reduce speaker bias
3. Experiment with a Platt scaling alternative to isotonic regression for calibration
4. Add a live feedback loop: when a market resolves, immediately update training data and trigger retraining

---
*Built at Qourex — github.com/sharathsaiK/kalshi-trading-bot*